# AdaBoost

AdaBoost melatih model secara berurutan. Setiap model baru memberi bobot lebih besar pada sampel yang salah diklasifikasi model sebelumnya. Hasil akhirnya adalah gabungan berbobot seluruh model.

## Tujuan

- Memahami mekanisme pembobotan sampel.
- Melihat pengaruh learning rate terhadap akurasi.
- Membandingkan AdaBoost dengan model tunggal.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier


In [ ]:
def get_accuracy(X_train, X_test, y_train, y_test, model):
    return {
        "test Accuracy": metrics.accuracy_score(y_test, model.predict(X_test)),
        "train Accuracy": metrics.accuracy_score(y_train, model.predict(X_train)),
    }


## Data churn pelanggan


In [ ]:
CHURN_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "ChurnData.csv"
)
churn_data = pd.read_csv(CHURN_URL)
churn_data = churn_data[
    ["tenure", "age", "address", "income", "ed", "employ", "equip",
     "callcard", "wireless", "churn"]
]
churn_data["churn"] = churn_data["churn"].astype("int")

features = churn_data[["tenure", "age", "address", "income", "ed", "employ", "equip"]]
target = churn_data["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=0
)
print("Train set", X_train.shape, y_train.shape)
print("Test set", X_test.shape, y_test.shape)


## Pohon tunggal sebagai pembanding


In [ ]:
stump = DecisionTreeClassifier(
    criterion="entropy", max_depth=1, random_state=0
).fit(X_train, y_train)
print(get_accuracy(X_train, X_test, y_train, y_test, stump))


## AdaBoost dengan stump

`DecisionTreeClassifier(max_depth=1)` disebut stump karena hanya punya satu percabangan. Stump dipakai sebagai model dasar agar setiap iterasi hanya memperbaiki kesalahan sederhana.


In [ ]:
boost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    learning_rate=1.0,
    random_state=0,
).fit(X_train, y_train)
print(get_accuracy(X_train, X_test, y_train, y_test, boost_model))


## Pengaruh learning rate

Learning rate kecil membuat kontribusi setiap model lebih halus, sehingga butuh lebih banyak estimator untuk mencapai akurasi tinggi.


In [ ]:
learning_rates = [0.2, 0.4, 0.6, 1.0]
estimator_counts = [10, 25, 50, 100]

curve = {}
for rate in learning_rates:
    values = []
    for count in estimator_counts:
        model = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1),
            n_estimators=count,
            learning_rate=rate,
            random_state=0,
        ).fit(X_train, y_train)
        values.append(metrics.accuracy_score(y_test, model.predict(X_test)))
    curve[rate] = values

pd.DataFrame(curve, index=estimator_counts)


In [ ]:
pd.DataFrame(curve, index=estimator_counts).plot(
    marker="o", figsize=(10, 6), linestyle="--"
)
plt.xlabel("Number of estimators")
plt.ylabel("Test accuracy")
plt.title("AdaBoost learning rate comparison")
plt.tight_layout()
plt.show()


## Grid search

Kombinasi jumlah estimator dan learning rate diuji dengan validasi silang.


In [ ]:
search = GridSearchCV(
    AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1), random_state=0
    ),
    param_grid={
        "n_estimators": [25, 50, 100],
        "learning_rate": [0.1, 0.5, 1.0],
    },
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)
search.best_score_, search.best_params_


In [ ]:
print(get_accuracy(X_train, X_test, y_train, y_test, search.best_estimator_))


## Data sel tumor


In [ ]:
CANCER_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "cell_samples.csv"
)
cell_data = pd.read_csv(CANCER_URL)
cell_data = cell_data[pd.to_numeric(cell_data["BareNuc"], errors="coerce").notnull()]

cell_features = cell_data[
    ["Clump", "UnifSize", "UnifShape", "MargAdh", "SingEpiSize",
     "BareNuc", "BlandChrom", "NormNucl", "Mit"]
].astype("int")
cell_target = cell_data["Class"].astype("int")

X_train, X_test, y_train, y_test = train_test_split(
    cell_features, cell_target, test_size=0.2, random_state=4
)
print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)


In [ ]:
cancer_search = GridSearchCV(
    AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1), random_state=0
    ),
    param_grid={"n_estimators": [25, 50, 100], "learning_rate": [0.5, 1.0]},
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)

cancer_search.best_score_, cancer_search.best_params_


In [ ]:
print(get_accuracy(X_train, X_test, y_train, y_test, cancer_search.best_estimator_))


## Ringkasan

AdaBoost memperbaiki akurasi stump tunggal secara bertahap. Learning rate kecil membutuhkan lebih banyak estimator, sedangkan learning rate besar bisa lebih cepat tetapi rawan overfitting pada data berderau.
